# Table 3 and the word set

Section 3 and Appendix A quote these numbers for the word lists, the backgrounds, the typography, the class names and centering.

Inputs: `lexicon/block*.txt`; `lexicon/words.parquet`, `lexicon/pairs.parquet` and `lexicon/audit.parquet` from `scripts/stimuli/wordlists.py`; `lexicon/manifest.parquet` from `scripts/stimuli/render.py`; `backgrounds/backgrounds.parquet` from `scripts/stimuli/backgrounds.py`; `lexicon/concepts.parquet` from `scripts/stimuli/class_names.py`; `concepts/manifest.parquet` and `concepts/images.parquet` from `scripts/stimuli/class_images.py`; `results/centering.parquet` from `scripts/analysis/table1_models.py`.

In [ ]:
import pandas as pd

from src import lexicon
from src.config import BACKGROUNDS, IMAGE_SIZE, data_relative
from src.slots import REPLICATES, slot_table
from src.tau import CONCEPT_IMAGES, CONCEPT_WORDS, manifest, word_order
from src.typography import (FONT_FAMILIES, PLACEMENTS, SIZE_BAND_NAMES, balance_report,
                            build_appearance_design, crop_safe_window, font_px)
from src.utils import setup_notebook
from src.utils.results import audit, concept_words, pairs, read, sweep_rows, words

setup_notebook()

W, P, A = words(), pairs(), audit()
gate = A.set_index("gate").value
V = lexicon.read_vocabulary()

## Table 3

The Words column counts distinct words, and the Pairs column counts pairs before the query direction is reversed.

In [ ]:
table3 = lexicon.list_summary(V)
table3

Each three-word item gives two pairs that share its first word.

In [ ]:
LIST_NAMES = {slug: title for slug, title, *_ in lexicon.LISTS}
relations = pd.DataFrame([
    {"List": LIST_NAMES[slug], "First word": a,
     "Partner": "one-edit word" if b == "spelling_twin" else b,
     "Pairs": int(P.relation.eq(relation).sum())}
    for slug, a, b, relation, _ in lexicon.PAIRS])
display(relations.set_index("List"))
three_word = relations[relations.List.isin([LIST_NAMES[slug] for slug, *_ in lexicon.LISTS[:3]])]
print(f"lists of three-word items with synonym pairs: "
      f"{three_word[three_word.Partner.eq('synonym')].List.nunique()} of {three_word.List.nunique()}")
print(f"words, names and pseudowords in more than one list: {V.word.duplicated().sum()}")

## Vocabulary

The noun and brand lists cross their categories or industries with spelling clusters in cells of equal size.

In [ ]:
pseudo = W.role == "pseudoword"
print(f"words and names: {(~pseudo).sum():,}; pseudowords: {pseudo.sum()}; total: {len(W):,}")
print(f"letters per word, name or pseudoword: {W.n_chars.min()} to {W.n_chars.max()}")
print(f"words, names and pseudowords not in lowercase: {(W.word != W.word.str.lower()).sum()}")
print(f"letters per pseudoword: {W.n_chars[pseudo].min()} to {W.n_chars[pseudo].max()}")
print(f"distinct words in the five relation lists: {(W.role == 'pair').sum()}; pairs: {len(P)}")
print("items per relation list:",
      ", ".join(str(V.item[V.list == slug].nunique()) for slug, *_ in lexicon.LISTS[:5]))
print(f"retrieval candidates per query: the other {len(word_order()) - 1:,} words, "
      f"names and pseudowords")
queried = set(P.a) | set(P.b) | set(W.word[W.role.isin(["category", "brand"])])
print(f"pseudowords among the pair words or in the noun and brand lists: "
      f"{W.word[pseudo].isin(queried).sum()}")

In [ ]:
GRIDS = (  # role, list, label, plural, audit gate
    ("category", "noun", "category", "categories", "categories/crossing"),
    ("brand", "brand", "industry", "industries", "brands/crossing"),
)
for role, name, label, plural, crossing in GRIDS:
    grid = W[W.role == role]
    cells = pd.crosstab(grid.category.rename(label), grid.ortho.rename("spelling cluster"))
    display(cells)
    per_label = ", ".join(str(n) for n in sorted(set(cells.sum(axis=1))))
    per_cell = ", ".join(str(n) for n in sorted(set(cells.to_numpy().ravel())))
    print(f"{name} list: {len(grid)} words in {len(cells)} {plural} of {per_label}, "
          f"{cells.shape[1]} spelling clusters, {per_cell} words in each {label} and cluster")
    print(f"Cramér's V between {label} and spelling cluster: {gate[crossing]:g}")

## Word frequency

The frequency target is Zipf 3, one occurrence per million words in `wordfreq`.

In [ ]:
TARGET = 3.0
below = W[W.zipf < TARGET].item_role.value_counts()
print(f"below the frequency target: {below['capital']} capitals, {below['category']} nouns, "
      f"{below['brand']} brand names")

zipf = W.set_index("word").zipf
for slug, name in (("synonym_one_edit", "synonym and one-edit"),
                   ("synonym_antonym", "synonym and antonym"), ("synonym_negation", "negation")):
    z = lexicon.item_table(V, slug).apply(lambda column: column.map(zipf))
    ratio = 10 ** (z.max(axis=1) - z.min(axis=1))
    print(f"median ratio of highest to lowest frequency within an item, {name} list: "
          f"{ratio.median():.2f}")

## Spelling within items

In the one-edit list, an insertion makes the one-edit word one letter longer than its anchor and a deletion makes it one letter shorter.

In [ ]:
one_edit = lexicon.item_table(V, "synonym_one_edit")
distance = pd.Series([lexicon.edit_distance(a, b)
                      for a, b in zip(one_edit.anchor, one_edit.spelling_twin)])
change = one_edit.spelling_twin.str.len() - one_edit.anchor.str.len()
longer = (one_edit.synonym.str.len() - one_edit.anchor.str.len()).mean()
print(f"one-edit words at edit distance one: {(distance == 1).sum()} of {len(one_edit)}")
print(f"substitutions {(change == 0).sum()}, insertions {(change == 1).sum()}, "
      f"deletions {(change == -1).sum()}")
print(f"synonyms are longer than their anchor by {longer:.2f} letters on average")
quoted = one_edit.set_index("spelling_twin").loc[["ain", "lan", "icc"]]
print("one-edit words ain, lan and icc, with their anchor and Zipf frequency: "
      + ", ".join(f"{word} ({row.anchor}, {zipf[word]:.2f})" for word, row in quoted.iterrows()))

## Letter overlap and relatives

The excess is the mean letter overlap within a meaning relation minus that of random length-matched word pairs.

In [ ]:
letters = A[A.gate.str.startswith("pairs/letters/")]
display(letters[["gate", "detail", "value"]])
print(f"largest excess letter overlap: {letters.value.max():.4f}")
print(f"meaning pairs that are inflectional or derivational relatives: "
      f"{gate['pairs/orthography']:.0f}")

## Exceptions to the relation labels

Appendix A names these negation-list items and this plural pair as exceptions to their list's relation.

In [ ]:
QUOTED_BASES = (("injury", "jury"), ("invent", "vent"), ("distract", "tract"),
                ("impending", "pending"), ("harmless", "harmful"), ("invaluable", "valuable"))
negation = lexicon.item_table(V, "synonym_negation")
listed_bases = set(zip(negation.negation, negation.base))
found = [f"{a}-{b}" for a, b in QUOTED_BASES if (a, b) in listed_bases]
print(f"negation and base of a negation-list item, {len(found)} of {len(QUOTED_BASES)}: "
      f"{', '.join(found)}")
plural = lexicon.item_table(V, "singular_plural")
print(f"plural-list items ala-alas: {(plural.singular.eq('ala') & plural.plural.eq('alas')).sum()}")

## Backgrounds and typography

A slot is one background with one typography setting, and every word is rendered on every slot.

In [ ]:
B = pd.read_parquet(BACKGROUNDS / "backgrounds.parquet")
print(f"backgrounds: {len(B)} photographs from {B.imagenet_label.nunique()} ImageNet classes; "
      f"largest OCR confidence {B.ocr_conf.max():.3f}")

_, _, side = crop_safe_window()
widest = W.loc[W.max_px.idxmax()]
print(f"crop-safe window: {side} of {IMAGE_SIZE} pixels; words that do not fit: {(~W.fits).sum()}")
print(f"widest word, at the large size in its widest font: {widest.word}, "
      f"{widest.max_px:.0f} pixels")
print("font sizes:", ", ".join(f"{band} {font_px(band)} pixels" for band in SIZE_BAND_NAMES))

In [ ]:
design = build_appearance_design()
slots = slot_table()
print(f"{len(FONT_FAMILIES)} fonts x {len(PLACEMENTS)} position bands = {len(design)} settings, "
      f"each on {REPLICATES} backgrounds: {len(slots)} slots on "
      f"{slots.background_index.nunique()} distinct backgrounds")
print("settings per size:", design["size"].value_counts().sort_index().to_dict())
print("settings per color:", design.color.value_counts().sort_index().to_dict())
print(f"settings with a backing panel: {(design.box == 'panel').sum()} of {len(design)}")

balance = balance_report(design)
display(balance)
inexact = balance[~balance.exact]
strongest = inexact.loc[inexact.cramers_v.idxmax()]
print(f"exactly balanced factor pairs: {balance.exact.sum()} of {len(balance)}")
print(f"strongest association among the others: {strongest.pair}, "
      f"Cramér's V {strongest.cramers_v:.2f}")

In [ ]:
images = manifest()
display(images.condition.value_counts().rename("images").to_frame())
print(f"images: {len(images):,}")
rendered = images[images.condition.ne("clean")].groupby(["word", "condition"]).slot.nunique()
complete = rendered.unstack().eq(len(slots)).all(axis=1).sum()
print(f"words, names and pseudowords with a word image and a control image on all "
      f"{len(slots)} backgrounds: {complete:,} of {len(W):,}")

## Class names and class images

The class names are rendered on the same slots as the lexical vocabulary, and the class images are the ImageNet-1k validation photographs not used as backgrounds.

In [ ]:
C = concept_words()
names = manifest(CONCEPT_WORDS)
class_images = manifest(CONCEPT_IMAGES)


def image_files(frame, index):
    """Each image's path below the data root, indexed by ``index``."""
    return frame.set_index(index).image_path.map(lambda path: str(data_relative(path)))


print(f"class names: {len(C)}; single lowercase words: {C.word.str.fullmatch('[a-z]+').sum()}")
lexical_backgrounds = image_files(images[images.condition.eq("clean")], "slot")
name_backgrounds = image_files(names[names.condition.eq("clean")], "slot")
name_rendered = names[names.condition.ne("clean")].groupby(["word", "condition"]).slot.nunique()
print(f"class-name backgrounds equal to the lexical vocabulary's, slot for slot: "
      f"{name_backgrounds.eq(lexical_backgrounds.reindex(name_backgrounds.index)).sum()} of "
      f"{len(name_backgrounds)}")
print(f"class names with a word image and a control image on all {len(slots)} backgrounds: "
      f"{name_rendered.unstack().eq(len(slots)).all(axis=1).sum()} of {len(C)}")
shared = sorted(set(C.word) & set(W.word))
key = ["word", "slot", "condition"]
lexical_files = image_files(images[images.word.isin(shared)], key)
name_files = image_files(names[names.word.isin(shared)], key)
same = name_files.eq(lexical_files.reindex(name_files.index)).groupby(level="word").all().sum()
print(f"class names also in the lexical vocabulary: {len(shared)}; "
      f"rendered from the same image files: {same} of {len(shared)}")
print(f"class images: {len(class_images):,} from {class_images.wnid.nunique():,} classes; "
      f"backgrounds among them: {class_images.key.isin(B.key).sum()}")

## Centering

The shared-mean share is the squared norm of the vocabulary mean of the uncentered word differences divided by their mean squared norm.

In [ ]:
centering = sweep_rows(read("centering"))
display(centering[["label", "shared_mean_share"]].set_index("label"))
lowest = centering.loc[centering.shared_mean_share.idxmin()]
highest = centering.loc[centering.shared_mean_share.idxmax()]
median = centering.shared_mean_share.median()
print(f"median over the {len(centering)} models: {median:.0%} ({median:.4f})")
print(f"lowest: {lowest.shared_mean_share:.0%} in {lowest.label}; "
      f"highest: {highest.shared_mean_share:.0%} in {highest.label}")